# HRNet Semantic Segmentation

Подключаем библиотки и загружаем датасет

In [1]:
import os
import numpy as np
import cv2
import glob
import re
import matplotlib.pyplot as plt
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from PIL import Image
from tqdm import tqdm

# from random.shuffle import train_test_split # Если нет sklearn, можно random.shuffle

# Локальные библиотеки
import grasp_utils # from grasp_utils import visualize_sample, debug_predict, visualize_grasp_pro, visualize_dataset_hedgehog, visualize_all_predictions
# Импортируем нашу заглушку
from models.Dict2Obj import Dict2Obj
# Импортируем саму архитектуру
from models.seg_hrnet import get_seg_model
# Импортируем датасет 
from datasets.dataset import JacquardDataset

## ВЫСТАВЛЯЕМ ОСНОВНЫЕ ПАРАМЕТРЫ МОДЕЛИ И ОБУЧЕНИЯ

In [ ]:
# ==========================================
# 🎛️ CONTROL CENTER (Настройки обучения)
# ==========================================

# Пути к данным и сохранениям
DATASET_PATH = 'datasets\Jacquard_V2'               # Путь к распакованному датасету Jacquard V2
EXPERIMENT_NAME = 'HRNet-W48_RGBD_Base'             # Имя текущего эксперимента
CHECKPOINT_DIR = f'checkpoints/{EXPERIMENT_NAME}'   # Папка для весов модели
VISUAL_DIR = f'visual_results/{EXPERIMENT_NAME}'    # Папка для картинок
LOG_DIR = f'runs/{EXPERIMENT_NAME}'                 # Папка для TensorBoard

# Создаем директории, если их нет
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(VISUAL_DIR, exist_ok=True)

# Гиперпараметры
IMAGE_SIZE = 1024        # Размер входного изображения (1024x1024 для HRNet-W48)
BATCH_SIZE = 4           # 4 - Настроено под 6 ГБ VRAM !!!!!!!!!!!!!!!!!!!!!!!!!!!!
NUM_EPOCHS = 20          # Общее количество эпох
LEARNING_RATE = 1e-4     # Начальная скорость обучения
NUM_WORKERS = 8          # 0 для стабильности на Windows

# Индекс изображения из датасета для дебаг-визуализации в конце каждой эпохи
DEBUG_INDEX = 1160 

Проверка вычислительного устройства

In [3]:
# Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Используется устройство: {device}")

[*] Используется устройство: cuda


In [4]:
# 1. Собираем все пути один раз
all_rgb_files = glob.glob(os.path.join(DATASET_PATH, '**', '*_RGB.png'), recursive=True)
all_paths = [f.replace('_RGB.png', '') for f in all_rgb_files]

Считываем папки датасета и разделяем на две части: для обучения и для валидации

In [ ]:
# 2. Перемешиваем и делим (например, 90% на 10%)
random.shuffle(all_paths)
split = int(len(all_paths) * 0.9)
train_paths = all_paths[:split]
val_paths = all_paths[split:]

# 3. Создаем объекты (теперь они увидят список и не будут сканировать диск заново)
train_dataset = JacquardDataset(train_paths, IMAGE_SIZE)
val_dataset = JacquardDataset(val_paths, IMAGE_SIZE)
print(int(len(all_paths)), "всего изображений. ", len(train_paths), "для тренировки, ", len(val_paths), "для валидации.")

51601 всего изображений.  46440 для тренировки,  5161 для валидации.
[*] Dataset инициализирован: 46440 объектов.
[*] Dataset инициализирован: 5161 объектов.


Создаём класс датасета 

In [8]:
# Инициализация датасета
print("[*] Подготовка данных...")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
# ... дальше инициализация модели и цикл обучения ...

[*] Подготовка данных...


AttributeError: 'JacquardDataset' object has no attribute 'image_paths'

In [ ]:
# Создаем объект
dataset = JacquardDataset(DATASET_PATH, use_ram=False)

# Берем первый пример
x, y = dataset[1606]

print(f"Вход (RGB-D): {x.shape}") # Должно быть [4, 480, 480]
print(f"Выход (Маски): {y.shape}") # Должно быть [4, 480, 480]

# Запускаем проверку
visualize_sample(x, y)

In [ ]:
# Допустим, 'model' — это загруженная HRNet-W18
# Оригинальный первый слой: Conv2d(3, 64, kernel_size=3, stride=2, padding=1)

def expand_input_layer(model):
    # Достаем параметры старого слоя
    old_layer = model.conv1 
    
    # Создаем новый слой с 4 входными каналами
    new_layer = nn.Conv2d(4, old_layer.out_channels, 
                          kernel_size=old_layer.kernel_size, 
                          stride=old_layer.stride, 
                          padding=old_layer.padding, 
                          bias=old_layer.bias is not None)
    
    # Копируем веса RGB
    with torch.no_grad():
        new_layer.weight[:, :3, :, :] = old_layer.weight
        # Четвертый канал (Depth) заполняем средним от RGB для стабильного старта
        new_layer.weight[:, 3, :, :] = old_layer.weight.mean(dim=1)
    
    model.conv1 = new_layer
    return model

In [ ]:
# Наш конфиг (можно оставить прямо здесь для удобства правок)
hrnet_w18_config = {
    'MODEL': {
        'PRETRAINED': '', # Добавили пустую строку, чтобы метод init_weights не сломался
        'EXTRA': {
            'FINAL_CONV_KERNEL': 1,
            'STAGE1': {'NUM_MODULES': 1, 'NUM_BRANCHES': 1, 'BLOCK': 'BOTTLENECK', 'NUM_BLOCKS': [4], 'NUM_CHANNELS': [64], 'FUSE_METHOD': 'SUM'},
            'STAGE2': {'NUM_MODULES': 1, 'NUM_BRANCHES': 2, 'BLOCK': 'BASIC', 'NUM_BLOCKS': [4, 4], 'NUM_CHANNELS': [18, 36], 'FUSE_METHOD': 'SUM'},
            'STAGE3': {'NUM_MODULES': 4, 'NUM_BRANCHES': 3, 'BLOCK': 'BASIC', 'NUM_BLOCKS': [4, 4, 4], 'NUM_CHANNELS': [18, 36, 72], 'FUSE_METHOD': 'SUM'},
            'STAGE4': {'NUM_MODULES': 3, 'NUM_BRANCHES': 4, 'BLOCK': 'BASIC', 'NUM_BLOCKS': [4, 4, 4, 4], 'NUM_CHANNELS': [18, 36, 72, 144], 'FUSE_METHOD': 'SUM'}
        },
        'ALIGN_CORNERS': False
    },
    'DATASET': {
        'NUM_CLASSES': 4 
    }
}

# Инициализируем конфиг через импортированную заглушку
cfg = Dict2Obj(hrnet_w18_config)

# Теперь GraspHRNet будет использовать этот cfg
class GraspHRNet(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.backbone = get_seg_model(config)
        
        # Расширяем вход до 4 каналов (как делали раньше)
        old_conv1 = self.backbone.conv1
        self.backbone.conv1 = nn.Conv2d(4, old_conv1.out_channels, 
                                        kernel_size=old_conv1.kernel_size, 
                                        stride=old_conv1.stride, 
                                        padding=old_conv1.padding, 
                                        bias=old_conv1.bias is not None)
        
        with torch.no_grad():
            self.backbone.conv1.weight[:, :3, :, :] = old_conv1.weight
            self.backbone.conv1.weight[:, 3, :, :] = old_conv1.weight.mean(dim=1)

    def forward(self, x):
        out = self.backbone(x)
        if out.shape[2:] != x.shape[2:]:
            out = F.interpolate(out, size=x.shape[2:], mode='bilinear', align_corners=False)
        return out

# Создаем модель
model = GraspHRNet(cfg)

In [ ]:
x_batch = torch.randn(1, 4, 512, 512) # Просто случайный шум для проверки
with torch.no_grad():
    predictions = model(x_batch)

print(f"Размер выхода: {predictions.shape}")

In [ ]:
class GraspLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # Заменяем BCELoss на BCEWithLogitsLoss
        # Эта функция сама применит Сигмоиду внутри!
        self.bce_logits = nn.BCEWithLogitsLoss() 
        self.mse = nn.MSELoss()

    def forward(self, pred, target):
        # pred: [B, 4, 512, 512]
        # target: [B, 4, 512, 512]
        
        # 1. Канал качества (теперь БЕЗ ручной сигмоиды)
        pred_q = pred[:, 0, :, :]
        target_q = target[:, 0, :, :]
        
        loss_q = self.bce_logits(pred_q, target_q)
        
        # 2. Маска для остальных каналов (тут сигмоида нужна только для порога)
        mask = torch.sigmoid(pred_q) > 0.5
        
        # 3. MSE для Sin, Cos, Width
        if mask.sum() > 0:
            # Выбираем значения только там, где есть захват
            p_others = pred[:, 1:, :, :].permute(0, 2, 3, 1)[mask]
            t_others = target[:, 1:, :, :].permute(0, 2, 3, 1)[mask]
            loss_angle_width = self.mse(p_others, t_others)
        else:
            loss_angle_width = pred.sum() * 0 # Пустой градиент, чтобы не ломать обучение

        return loss_q + loss_angle_width

In [ ]:
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast

# 1. Переносим модель на видеокарту
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 2. Оптимизатор (AdamW обычно стабильнее для трансформеров и тяжелых бэкбонов)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# 3. Функция потерь (берем ту, что обсудили выше)
criterion = GraspLoss().to(device)

# 4. Скалер для mixed precision (чтобы float16 не "взрывался")
scaler = torch.amp.GradScaler('cuda')

print(f"Используем устройство: {device}")
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU не найден")

In [ ]:
from torch.utils.data import DataLoader

# Укажи путь к своему датасету
dataset_path = 'Jacquard_Dataset_V2' 

train_dataset = JacquardDataset(dataset_path)

# batch_size=4 для начала, чтобы точно влезло в 6ГБ памяти
# num_workers=0 важен для Windows, чтобы не было ошибок доступа
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)

print(f"Загрузчик готов. Батчей в эпохе: {len(train_loader)}")

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(loader, desc="Обучение")
    for images, targets in pbar:
        images = images.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        
        # Включаем автоматическую смешанную точность
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, targets)
            
        # Обратный проход через скалер
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    return running_loss / len(loader)

In [ ]:
# Запусти для первого изображения
debug_predict(model, train_dataset, device, index=1160)

## ЦИКЛ ОБУЧЕНИЯ

In [ ]:
# 1. Функция поиска последнего чекпоинта (обновленная)
def load_latest_checkpoint(model, checkpoint_dir):
    checkpoints = glob.glob(os.path.join(checkpoint_dir, "epoch_*.pth"))
    if not checkpoints:
        print("[*] Начинаем обучение с нуля.")
        return 0
    epochs = [int(re.findall(r'epoch_(\d+)', f)[0]) for f in checkpoints]
    latest_epoch = max(epochs)
    weights_path = os.path.join(checkpoint_dir, f"epoch_{latest_epoch}.pth")
    model.load_state_dict(torch.load(weights_path, weights_only=True))
    print(f">>> Подхвачены веса эпохи {latest_epoch} из {weights_path}")
    return latest_epoch

In [ ]:
# 2. Инициализация обучения
writer = SummaryWriter(LOG_DIR)
start_epoch = load_latest_checkpoint(model, CHECKPOINT_DIR)

# 3. Ультимативный цикл обучения
for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Эпоха {epoch+1}/{NUM_EPOCHS}")
    
    for i, (images, targets) in enumerate(pbar):
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        
        # Записываем лосс в TensorBoard
        if i % 10 == 0:
            writer.add_scalar('Loss/train_batch', loss.item(), epoch * len(train_loader) + i)
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    # Средний лосс за эпоху
    avg_loss = running_loss / len(train_loader)
    writer.add_scalar('Loss/epoch', avg_loss, epoch)
    
    # Сохранение весов в новую папку
    save_path = os.path.join(CHECKPOINT_DIR, f'epoch_{epoch+1}.pth')
    torch.save(model.state_dict(), save_path)
    
    # Визуализация и авто-сохранение картинок
    print(f"\n[*] Эпоха {epoch+1} завершена. Сохраняем визуализацию...")
    debug_predict(model, train_dataset, device, index=DEBUG_INDEX, save_dir=VISUAL_DIR)

## РЕАЛЬНЫЕ ИЗОБРАЖЕНИЯ

In [ ]:
# 1. Настройки
INPUT_FOLDER = 'my_photos'       # Папка с твоими фото
OUTPUT_FOLDER = 'test_results'   # Куда сохранять результат
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

model.eval()

# Получаем список всех изображений
image_paths = glob.glob(os.path.join(INPUT_FOLDER, "*.jpg")) + \
              glob.glob(os.path.join(INPUT_FOLDER, "*.png"))

def process_real_photo(img_path):
    # Загрузка и ресайз
    raw_img = cv2.imread(img_path)
    raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
    original_h, original_w = raw_img.shape[:2]
    
    # Приводим к 512x512
    img_resized = cv2.resize(raw_img, (512, 512))
    
    # Подготовка тензора (RGB)
    img_tensor = img_resized.astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(img_tensor).permute(2, 0, 1) # [3, 512, 512]
    
    # ДОБАВЛЯЕМ 4-Й КАНАЛ (DEPTH)
    # Создаем "фейковую" глубину (просто заполняем средним значением)
    dummy_depth = torch.ones((1, 512, 512)) * 0.5
    full_input = torch.cat([img_tensor, dummy_depth], dim=0).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(full_input)
        
        q = torch.sigmoid(output[0, 0, :, :]).cpu().numpy()
        sin = output[0, 1, :, :].cpu().numpy()
        cos = output[0, 2, :, :].cpu().numpy()
        width = output[0, 3, :, :].cpu().numpy() * 150
        
    # Находим лучший захват
    y, x = np.unravel_index(np.argmax(q), q.shape)
    angle = np.arctan2(sin[y, x], cos[y, x]) / 2
    
    # Визуализация
    plt.figure(figsize=(6, 6))
    plt.imshow(img_resized)
    
    # Стрелка и ширина
    dx, dy = np.cos(angle) * 40, np.sin(angle) * 40
    plt.arrow(x, y, dx, dy, color='red', head_width=12)
    
    wx, wy = np.cos(angle + np.pi/2) * (width[y, x]/2), np.sin(angle + np.pi/2) * (width[y, x]/2)
    plt.plot([x - wx, x + wx], [y - wy, y + wy], color='lime', linewidth=4)
    
    plt.title(f"Conf: {q[y, x]:.2f} | Angle: {np.degrees(angle):.1f}°")
    plt.axis('off')
    
    # Сохраняем
    save_path = os.path.join(OUTPUT_FOLDER, "res_" + os.path.basename(img_path))
    plt.savefig(save_path)
    plt.show()
    print(f"Обработано: {img_path} -> Уверенность: {q[y, x]:.4f}")


In [ ]:
# Запуск обработки
for path in image_paths:
    process_real_photo(path)

## ВИЗУАЛИЗАЦИЯ

In [ ]:
visualize_grasp_pro(model, train_dataset, device, index=1160)

In [ ]:
debug_predict(model, train_dataset, device, index=1160)

In [ ]:
visualize_dataset_hedgehog(train_dataset, index=1606, step=100)

In [ ]:
visualize_all_predictions(model, train_dataset, device, index=25, threshold=0.1, step=8)

tensorboard --logdir=runs

3e1be313618251d6bbed0fb2b80f6c79 - прикольная картинка с телескопом в датасете


3eee9754c2b6877a50c6c4576c5d0002 - электрогитара

номер 303 - динозавр

25 - нормальный стул

3256 - стул какой-то

1160 - подходит